# Tái hiện Kết quả Chính (Main Experiments)

**Bài báo:** *Identifying and Clustering Counter Relationships of Team Compositions in PvP Games for Efficient Balance Analysis* (TMLR 09/2024)

**Phụ trách:** Thành viên 2 — Core Algorithm Implementer

Notebook này tái hiện các thực nghiệm chính của bài báo trên tập dữ liệu League of Legends (Kaggle), bao gồm:
1. Huấn luyện Neural Rating Table (NRT) theo mô hình Bradley-Terry
2. Tính Residual Win Value ($W_{res}$) thể hiện mối quan hệ khắc chế
3. Huấn luyện Neural Counter Table (NCT) với Vector Quantization
4. Đánh giá Strength Relation Accuracy (metric chính của paper — Table 1, 2)
5. Đánh giá chất lượng phân cụm (Silhouette, Davies-Bouldin, Calinski-Harabasz — theo yêu cầu đề bài §3.2.4)
6. Bảng so sánh kết quả Paper vs. Nhóm (theo yêu cầu đề bài §3.2.3a)
7. Phân tích Codebook Utilization (Section 4.3 trong paper)
8. Trực quan hóa t-SNE và Counter Table Heatmap

In [ ]:
import os
import sys
# Thêm thư mục gốc vào đường dẫn để import src
sys.path.append(os.path.abspath('..'))

import torch
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split

from src.model import NRT, NCT
from src.metrics import evaluate_clustering, compute_strength_accuracy, codebook_utilization
from src.utils import load_data, preprocess_data, extract_team_features

# ===== Đặt random seed cố định (theo yêu cầu đề bài §3.2.2) =====
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {DEVICE}")

## 1. Chuẩn bị Dữ liệu

Sử dụng tập dữ liệu League of Legends từ Kaggle (`data/games.csv`). Dữ liệu được biểu diễn dưới dạng **multi-hot binary vector** cho mỗi đội hình (composition), tương tự cách bài báo mô tả ở Appendix A.4:
> *"League of Legends: 136-dimensional binary vector for champion representation."*

Lưu ý: Dataset của nhóm có thể có số tướng khác so với paper (do nguồn dữ liệu khác nhau), nên chiều vector sẽ được tính tự động.

In [ ]:
print("Đang tải dữ liệu thực tế từ data/games.csv...")
df_raw = load_data('../data/games.csv')

if df_raw.empty:
    raise ValueError("Không tìm thấy dữ liệu! Hãy đảm bảo dataset nằm trong thư mục data/")

df_processed = preprocess_data(df_raw)

# Xác định số chiều đặc trưng tự động (thay vì hardcode)
all_champ_ids = set()
for t in df_processed['team_1']:
    all_champ_ids.update(t)
for t in df_processed['team_2']:
    all_champ_ids.update(t)
NUM_CHAMPIONS = max(all_champ_ids) + 1
NUM_UNIQUE = len(all_champ_ids)

print(f"Số tướng duy nhất trong dataset: {NUM_UNIQUE}")
print(f"Dimension đầu vào (max ID + 1): {NUM_CHAMPIONS}")

# Trích xuất features: Multi-hot Encoding
features = extract_team_features(df_processed, num_champions=NUM_CHAMPIONS)

# Labels: 1.0 nếu Team 1 thắng, 0.0 nếu Team 1 thua
labels = np.where(df_processed['winner'].values == 1, 1.0, 0.0).astype(np.float32)

print(f"Kích thước Features: {features.shape}  →  (N_matches, 2_teams, {NUM_CHAMPIONS}_champions)")
print(f"Tổng số trận đấu: {len(labels)}")
print(f"Tỷ lệ Team 1 thắng: {labels.mean():.4f}")

## 2. Chia tập Train / Test và Data Augmentation

Theo paper Section 4.2:
> *"All models are [...] trained for 100 epochs on datasets using **5-fold cross-validation**."*
> *"the random swapping of compositions in a match and corresponding win outcome"*

Ta sử dụng **train/test split 80/20** (tương đương 1 fold của 5-fold CV) và áp dụng **data augmentation** bằng cách hoán đổi vị trí hai đội (swap Team A ↔ Team B, đảo nhãn thắng/thua).

In [ ]:
# Train/Test split (80/20)
n_samples = len(labels)
indices = np.arange(n_samples)
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=SEED)

# Data Augmentation: Swap Team A <-> Team B, đảo win label
# Theo paper: "random swapping of compositions in a match and corresponding win outcome"
def augment_data(feats, labs):
    """Tạo bản sao với Team A và Team B hoán đổi, nhãn thắng/thua đảo ngược."""
    swapped_feats = feats[:, [1, 0], :]  # Đảo thứ tự team
    swapped_labs = 1.0 - labs             # Đảo nhãn (thắng→thua, thua→thắng)
    aug_feats = np.concatenate([feats, swapped_feats], axis=0)
    aug_labs = np.concatenate([labs, swapped_labs], axis=0)
    return aug_feats, aug_labs

# Augment CHỈ tập train (không augment test để đánh giá công bằng)
train_features_np = features[train_idx]
train_labels_np = labels[train_idx]
train_features_aug, train_labels_aug = augment_data(train_features_np, train_labels_np)

# Chuyển sang tensor
train_features_tensor = torch.tensor(train_features_aug, dtype=torch.float32)
train_labels_tensor = torch.tensor(train_labels_aug, dtype=torch.float32)
test_features_tensor = torch.tensor(features[test_idx], dtype=torch.float32)
test_labels_tensor = torch.tensor(labels[test_idx], dtype=torch.float32)
# Tensor toàn bộ (dùng cho evaluation và visualization sau này)
all_features_tensor = torch.tensor(features, dtype=torch.float32)
all_labels_tensor = torch.tensor(labels, dtype=torch.float32)

print(f"Train set (gốc):       {len(train_idx)} trận")
print(f"Train set (augmented): {len(train_features_tensor)} trận (x2 từ swap)")
print(f"Test set:              {len(test_idx)} trận")

## 3. Phase 1: Huấn luyện Neural Rating Table (NRT)

Huấn luyện mô hình NRT trên Ground Truth (Win/Loss) để học Rating tự nhiên của các đội hình, thông qua Bradley-Terry Model (Section 3.1):

$$P_{\text{expected}}(A > B) = \frac{R_\theta(c_A)}{R_\theta(c_A) + R_\theta(c_B)}$$

**Hyperparameters theo paper Section 4.2:**
- Epochs: **100**
- Learning rate: **0.00025 → 0** (linear decay)
- Optimizer: **Adam**

In [ ]:
# ===== Hyperparameters theo paper Section 4.2 =====
NRT_EPOCHS = 100
NRT_LR = 0.00025
BATCH_SIZE = 256

nrt_model = NRT(input_dim=NUM_CHAMPIONS).to(DEVICE)
nrt_optimizer = optim.Adam(nrt_model.parameters(), lr=NRT_LR)
# Linear decay từ 0.00025 → 0 theo paper:
# "A linear decay learning rate from 0.00025 to 0 over 100 epochs"
nrt_scheduler = optim.lr_scheduler.LambdaLR(
    nrt_optimizer, lr_lambda=lambda epoch: max(0.0, 1.0 - epoch / NRT_EPOCHS)
)

train_dataset = TensorDataset(train_features_tensor, train_labels_tensor)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print("--- BẮT ĐẦU TRAINING NRT ---")
nrt_losses = []
nrt_model.train()
for epoch in range(NRT_EPOCHS):
    total_loss = 0
    for batch_X, batch_y in train_dataloader:
        batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
        nrt_optimizer.zero_grad()
        
        comp_A = batch_X[:, 0, :]
        comp_B = batch_X[:, 1, :]
        expected_win_A, _, _ = nrt_model(comp_A, comp_B)
        
        # NRT Loss: MSE giữa Bradley-Terry prediction và Actual Win/Loss
        loss = F.mse_loss(expected_win_A.squeeze(), batch_y)
        loss.backward()
        nrt_optimizer.step()
        total_loss += loss.item()
    
    nrt_scheduler.step()
    avg_loss = total_loss / len(train_dataloader)
    nrt_losses.append(avg_loss)
    
    if (epoch + 1) % 20 == 0:
        lr_now = nrt_optimizer.param_groups[0]['lr']
        print(f"  Epoch {epoch+1:3d}/{NRT_EPOCHS} — Loss: {avg_loss:.6f} — LR: {lr_now:.6f}")

print("--- NRT TRAINING HOÀN TẤT ---")

# Plot training loss
plt.figure(figsize=(10, 4))
plt.plot(nrt_losses, color='#2196F3', linewidth=1.5)
plt.title('NRT Training Loss', fontsize=13)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Tạo Target Residual ($W_{res}$)

Mối quan hệ khắc chế (Counter Relationship) được biểu diễn bằng **Residual Win Value** — phần dư sau khi trừ đi win rate kỳ vọng từ sức mạnh đơn thuần (Eq. 7):

$$W_{res}(c_A, c_B | R_\theta) = W_m - \frac{R_\theta(c_A)}{R_\theta(c_A) + R_\theta(c_B)}$$

Nếu $W_{res} > 0$: đội A **khắc chế** đội B (thắng nhiều hơn kỳ vọng).
Nếu $W_{res} < 0$: đội A **bị khắc chế** bởi đội B.

In [ ]:
nrt_model.eval()
with torch.no_grad():
    # Tính residual cho TOÀN BỘ dữ liệu
    all_comp_A = all_features_tensor[:, 0, :].to(DEVICE)
    all_comp_B = all_features_tensor[:, 1, :].to(DEVICE)
    all_expected, _, _ = nrt_model(all_comp_A, all_comp_B)
    
    # Eq. (7): W_res = W_m - R(A)/(R(A)+R(B))
    all_residual = all_labels_tensor.unsqueeze(1).to(DEVICE) - all_expected

    # Tính residual cho tập train (phải augment tương ứng)
    train_comp_A = train_features_tensor[:, 0, :].to(DEVICE)
    train_comp_B = train_features_tensor[:, 1, :].to(DEVICE)
    train_expected, _, _ = nrt_model(train_comp_A, train_comp_B)
    train_residual = train_labels_tensor.unsqueeze(1).to(DEVICE) - train_expected

print(f"Residual Target (toàn bộ): {all_residual.shape}")
print(f"Residual Target (train aug): {train_residual.shape}")
print(f"Residual stats — Mean: {all_residual.mean():.4f}, Std: {all_residual.std():.4f}")
print(f"Residual range: [{all_residual.min():.4f}, {all_residual.max():.4f}]")

## 5. Phase 2: Huấn luyện Neural Counter Table (NCT) với Vector Quantization

Huấn luyện Siamese NCT để dự đoán phần dư $W_{res}$ qua không gian VQ rời rạc (Section 3.2.2, Figure 2).

**Tổng loss theo Eq. (12):**

$$\mathcal{L} = \mathcal{L}_{res} + \mathcal{L}_{codebook} + \beta_N \cdot \mathcal{L}_{commit} + \beta_M \cdot \mathcal{L}_{mean}$$

Trong đó:
- $\mathcal{L}_{res}$: Reconstruction loss cho residual win value (Eq. 9)
- $\mathcal{L}_{codebook}$: Đẩy codebook vectors về phía encoder output
- $\mathcal{L}_{commit}$: Đẩy encoder output về phía codebook ($\beta_N = 0.01$)
- $\mathcal{L}_{mean}$: VQ Mean Loss chống sụp đổ codebook ($\beta_M = 0.25$, Eq. 11)

**Hyperparameters:** $M = 9$, embedding\_dim $= 128$, epochs $= 100$, LR decay $0.00025 \to 0$.

In [ ]:
# ===== Hyperparameters =====
M_CLUSTERS = 9       # Số cụm (paper Table 2 báo cáo M=3,9,27,81)
NCT_EPOCHS = 100
NCT_LR = 0.00025
EMBEDDING_DIM = 128

nct_model = NCT(
    input_dim=NUM_CHAMPIONS,
    num_embeddings=M_CLUSTERS,
    embedding_dim=EMBEDDING_DIM
).to(DEVICE)
nct_optimizer = optim.Adam(nct_model.parameters(), lr=NCT_LR)
nct_scheduler = optim.lr_scheduler.LambdaLR(
    nct_optimizer, lr_lambda=lambda epoch: max(0.0, 1.0 - epoch / NCT_EPOCHS)
)

beta_N = nct_model.vq.beta_N  # 0.01 (Section 4.3, Table 3)
beta_M = nct_model.vq.beta_M  # 0.25 (Section 4.3, Table 4)

nct_train_dataset = TensorDataset(train_features_tensor, train_residual.cpu())
nct_train_dataloader = DataLoader(nct_train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Hyperparameters: βN={beta_N}, βM={beta_M}, M={M_CLUSTERS}, dim={EMBEDDING_DIM}")
print("\n--- BẮT ĐẦU TRAINING NCT ---")

nct_loss_history = {'total': [], 'res': [], 'codebook': [], 'commit': [], 'mean': []}
nct_model.train()

for epoch in range(NCT_EPOCHS):
    ep_loss = {'total': 0, 'res': 0, 'codebook': 0, 'commit': 0, 'mean': 0}
    
    for batch_X, batch_res in nct_train_dataloader:
        batch_X, batch_res = batch_X.to(DEVICE), batch_res.to(DEVICE)
        nct_optimizer.zero_grad()
        
        comp_A = batch_X[:, 0, :]
        comp_B = batch_X[:, 1, :]
        
        # Forward pass — trả về 6 giá trị theo API mới
        residual_pred, loss_codebook, loss_commit, loss_mean, _, _ = nct_model(comp_A, comp_B)
        
        # Reconstruction loss (Eq. 9)
        loss_res = F.mse_loss(residual_pred, batch_res)
        
        # Tổng loss theo Eq. (12):
        # Encoder nhận: ∂Lres (via gradient copy) + βN × ∂Lcommit
        # Codebook nhận: ∂Lcodebook + βM × ∂Lmean
        # Decoder nhận: ∂Lres
        loss = loss_res + loss_codebook + beta_N * loss_commit + beta_M * loss_mean
        
        loss.backward()
        nct_optimizer.step()
        
        ep_loss['total'] += loss.item()
        ep_loss['res'] += loss_res.item()
        ep_loss['codebook'] += loss_codebook.item()
        ep_loss['commit'] += loss_commit.item()
        ep_loss['mean'] += loss_mean.item()
    
    nct_scheduler.step()
    n_batches = len(nct_train_dataloader)
    for k in ep_loss:
        nct_loss_history[k].append(ep_loss[k] / n_batches)
    
    if (epoch + 1) % 20 == 0:
        print(f"  Epoch {epoch+1:3d}/{NCT_EPOCHS} — "
              f"Total: {nct_loss_history['total'][-1]:.6f}, "
              f"Lres: {nct_loss_history['res'][-1]:.6f}, "
              f"Lcb: {nct_loss_history['codebook'][-1]:.6f}, "
              f"Lcm: {nct_loss_history['commit'][-1]:.6f}, "
              f"Lmean: {nct_loss_history['mean'][-1]:.6f}")

print("--- NCT TRAINING HOÀN TẤT ---")

# Plot NCT losses
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(nct_loss_history['total'], color='#E91E63', linewidth=1.5)
axes[0].set_title('NCT Total Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(nct_loss_history['res'], label='$L_{res}$', linewidth=2)
axes[1].plot(nct_loss_history['codebook'], label='$L_{codebook}$', alpha=0.7)
axes[1].plot(nct_loss_history['commit'], label='$L_{commit}$', alpha=0.7)
axes[1].plot(nct_loss_history['mean'], label='$L_{mean}$', alpha=0.7)
axes[1].set_title('NCT Loss Components')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Đánh giá Strength Relation Accuracy (Metric chính của Paper)

Theo Section 4.2 và A.4.1 của bài báo, đánh giá bằng **Strength Relation Classification Accuracy**:
- Nếu win value > 0.501 → dự đoán "stronger"
- Nếu win value < 0.499 → dự đoán "weaker"
- Nếu 0.499 ≤ win value ≤ 0.501 → dự đoán "same"

**NRT** dự đoán: $\hat{W} = R(A) / (R(A) + R(B))$

**NCT** dự đoán (Eq. 13): $\hat{W} = R(A) / (R(A) + R(B)) + W_{res}(c_A, c_B)$

In [ ]:
def evaluate_model_accuracy(model_type, nrt_model, nct_model, features, labels, device):
    """
    Tính Strength Relation Classification Accuracy cho NRT hoặc NCT.
    Theo Section A.4.1 của bài báo.
    """
    nrt_model.eval()
    if nct_model is not None:
        nct_model.eval()
    
    with torch.no_grad():
        comp_A = features[:, 0, :].to(device)
        comp_B = features[:, 1, :].to(device)
        
        if model_type == 'NRT':
            pred_win, _, _ = nrt_model(comp_A, comp_B)
            pred_win = pred_win.squeeze().cpu().numpy()
        elif model_type == 'NCT':
            expected_win, _, _ = nrt_model(comp_A, comp_B)
            residual_pred, _, _, _, _, _ = nct_model(comp_A, comp_B)
            # Eq. (13): W_θ(A,B) = R(A)/(R(A)+R(B)) + W_res(A,B)
            pred_win = (expected_win.squeeze() + residual_pred.squeeze()).cpu().numpy()
            pred_win = np.clip(pred_win, 0, 1)
    
    true_win = labels.numpy() if isinstance(labels, torch.Tensor) else labels
    accuracy = compute_strength_accuracy(pred_win, true_win)
    return accuracy

# ===== Đánh giá trên Train và Test =====
# Dùng dữ liệu GỐC (không augment) để đánh giá công bằng
train_orig_features = torch.tensor(features[train_idx], dtype=torch.float32)
train_orig_labels = torch.tensor(labels[train_idx], dtype=torch.float32)

nrt_train_acc = evaluate_model_accuracy('NRT', nrt_model, None, train_orig_features, train_orig_labels, DEVICE)
nrt_test_acc  = evaluate_model_accuracy('NRT', nrt_model, None, test_features_tensor, test_labels_tensor, DEVICE)

nct_train_acc = evaluate_model_accuracy('NCT', nrt_model, nct_model, train_orig_features, train_orig_labels, DEVICE)
nct_test_acc  = evaluate_model_accuracy('NCT', nrt_model, nct_model, test_features_tensor, test_labels_tensor, DEVICE)

print("=" * 55)
print("  STRENGTH RELATION ACCURACY (%) — Metric của Paper")
print("=" * 55)
print(f"{'Model':<15} {'Train Acc':>12} {'Test Acc':>12}")
print("-" * 55)
print(f"{'NRT':<15} {nrt_train_acc:>11.1f}% {nrt_test_acc:>11.1f}%")
print(f"{'NCT M=' + str(M_CLUSTERS):<15} {nct_train_acc:>11.1f}% {nct_test_acc:>11.1f}%")
print("-" * 55)

## 7. Đánh giá Chất lượng Phân cụm (Clustering Metrics — Yêu cầu Đề bài §3.2.4)

Trích xuất đặc trưng ẩn ($z_e$) và nhãn cụm (Codebook Index) từ NCT encoder + VQ layer.

Báo cáo 3 độ đo không giám sát theo yêu cầu đề bài:
1. **Silhouette Score** (càng gần 1 càng tốt)
2. **Davies-Bouldin Index** (càng nhỏ càng tốt)
3. **Calinski-Harabasz Index** (càng lớn càng tốt)

Đánh giá trên **cả Team A và Team B** (Siamese architecture sử dụng shared encoder).

In [ ]:
nct_model.eval()
with torch.no_grad():
    # Lấy latent features và cluster labels cho CẢ HAI team (không chỉ Team A)
    all_comp_A = all_features_tensor[:, 0, :].to(DEVICE)
    all_comp_B = all_features_tensor[:, 1, :].to(DEVICE)
    
    z_e_A = nct_model.encoder(all_comp_A)
    z_e_B = nct_model.encoder(all_comp_B)
    _, _, _, _, idx_A = nct_model.vq(z_e_A)
    _, _, _, _, idx_B = nct_model.vq(z_e_B)
    
    # Gộp cả hai team để đánh giá clustering tổng thể
    all_latent = torch.cat([z_e_A, z_e_B], dim=0).cpu().numpy()
    all_cluster_labels = torch.cat([idx_A, idx_B], dim=0).cpu().numpy().flatten()

# Đánh giá clustering
metrics = evaluate_clustering(all_latent, all_cluster_labels)
used_M = codebook_utilization(all_cluster_labels, M_CLUSTERS)

print("=" * 55)
print("  CLUSTERING METRICS — Yêu cầu đề bài §3.2.4")
print("=" * 55)
for k, v in metrics.items():
    print(f"  {k:<22}: {v:.4f}")
print(f"  {'Codebook Utilization':<22}: {used_M}/{M_CLUSTERS} (Used M / Total M)")
print("=" * 55)

## 8. Codebook Utilization Analysis (Section 4.3)

Theo paper Section 4.3:
> *"We introduced a VQ Mean Loss to maximize the utilized M."*

Kiểm tra xem VQ Mean Loss ($\beta_M = 0.25$) có giúp tất cả $M$ codebook vectors đều được sử dụng hay không. Paper báo cáo kết quả ở Table 3 và Table 4.

In [ ]:
# Phân bố cluster labels
unique_labels, counts = np.unique(all_cluster_labels, return_counts=True)

print(f"Tổng số codebook vectors (M): {M_CLUSTERS}")
print(f"Số vectors được sử dụng (Used M): {used_M}")
print(f"\nPhân bố từng cluster:")
for label, count in zip(unique_labels, counts):
    pct = count / len(all_cluster_labels) * 100
    bar = '█' * int(pct / 2)
    print(f"  Cluster {label:2d}: {count:7d} samples ({pct:5.1f}%) {bar}")

# Bar chart phân bố
plt.figure(figsize=(10, 4))
bars = plt.bar(unique_labels, counts, color=plt.cm.tab10(np.arange(len(unique_labels)) % 10),
               edgecolor='white', linewidth=0.5)
plt.title(f'Phân bố Codebook Clusters (Used M = {used_M}/{M_CLUSTERS})', fontsize=13)
plt.xlabel('Cluster ID')
plt.ylabel('Số lượng samples')
plt.xticks(unique_labels)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 9. Bảng So sánh Kết quả Paper vs. Nhóm (Yêu cầu §3.2.3a)

Theo yêu cầu đề bài §3.2.3a:
> *"Tổng hợp kết quả vào bảng so sánh giữa: (i) số liệu được báo cáo trong bài báo; (ii) số liệu từ cài đặt của nhóm. Tính độ lệch tương đối cho từng chỉ số."*

**Lưu ý quan trọng:** Dataset của paper và dataset của nhóm KHÁC NHAU:
- Paper sử dụng dữ liệu LOL riêng (136 champions, nguồn không công khai)
- Nhóm sử dụng Kaggle games.csv (nguồn công khai, số champions khác)

Do đó, sai lệch kết quả là **kỳ vọng** và sẽ được phân tích ở phần sai lệch.

In [ ]:
# Kết quả từ Paper (Table 2, cho League of Legends)
paper_results = {
    'NRT_train': 88.2,
    'NRT_test': 51.1,
    'NCT_M9_train': 91.6,
    'NCT_M9_test': 50.9,
}

# Kết quả của nhóm
group_results = {
    'NRT_train': nrt_train_acc,
    'NRT_test': nrt_test_acc,
    'NCT_M9_train': nct_train_acc,
    'NCT_M9_test': nct_test_acc,
}

# Tính độ lệch tương đối
def relative_deviation(paper_val, group_val):
    if paper_val == 0:
        return float('inf')
    return (group_val - paper_val) / paper_val * 100

print("=" * 75)
print("  BẢNG SO SÁNH KẾT QUẢ: PAPER vs. NHÓM")
print("  Metric: Strength Relation Classification Accuracy (%)")
print("=" * 75)
print(f"{'Thực nghiệm':<25} {'Paper':>10} {'Nhóm':>10} {'Độ lệch (%)':>15}")
print("-" * 75)

rows = [
    ('NRT — Train', 'NRT_train'),
    ('NRT — Test', 'NRT_test'),
    (f'NCT M={M_CLUSTERS} — Train', 'NCT_M9_train'),
    (f'NCT M={M_CLUSTERS} — Test', 'NCT_M9_test'),
]

for label, key in rows:
    p = paper_results[key]
    g = group_results[key]
    dev = relative_deviation(p, g)
    print(f"  {label:<23} {p:>9.1f}% {g:>9.1f}% {dev:>+13.1f}%")

print("-" * 75)
print()

# Bảng kết quả Clustering Metrics (chỉ nhóm — paper không báo cáo metrics này)
print("=" * 55)
print("  CLUSTERING METRICS (Không giám sát — Chỉ nhóm)")
print("=" * 55)
print(f"  {'Metric':<25} {'Giá trị':>12}")
print("-" * 55)
for k, v in metrics.items():
    print(f"  {k:<25} {v:>12.4f}")
print(f"  {'Used M / Total M':<25} {used_M:>5}/{M_CLUSTERS:<5}")
print("=" * 55)

## 10. Phân tích Sai lệch (Yêu cầu §3.2.3b)

Theo yêu cầu đề bài §3.2.3b:
> *"Nếu kết quả tái hiện khác với bài báo, phân tích và giải thích nguyên nhân có thể (siêu tham số chưa được nêu rõ, phiên bản thư viện, khởi tạo ngẫu nhiên, …)"*

### Phân tích nguyên nhân sai lệch:

1. **Dataset khác nhau:** Paper sử dụng dữ liệu LOL riêng (136 champions), nhóm sử dụng Kaggle games.csv. Số tướng, phân bố trận đấu, và phiên bản game khác nhau dẫn đến không gian composition khác biệt đáng kể.

2. **Kích thước không gian đầu vào:** Paper dùng 136-dim binary vector, dataset nhóm có thể lớn hơn nhiều → encoder phải xử lý input thưa hơn (sparse), ảnh hưởng đến khả năng học.

3. **Overfitting trầm trọng ở LOL:** Paper thừa nhận ở Section 4.2 rằng LOL có *"clear overfitting"* với test accuracy chỉ ~51% (gần random). Nguyên nhân là không gian composition quá lớn (ban/pick phase tạo ra quá nhiều đội hình khả thi) nên mô hình không generalize được sang unseen matchups.

4. **Kiến trúc mạng chính xác:** Paper chỉ mô tả kiến trúc qua hình vẽ (Figure 16-17), không nêu rõ từng hyperparameter (hidden dim, activation slopes). Nhóm đã sử dụng kiến trúc 4-layer FC, 128 hidden units, LeakyReLU(0.01) dựa trên diễn giải hình vẽ.

5. **Số run:** Paper tính trung bình 5 random seeds, nhóm chỉ chạy 1 run do giới hạn thời gian.

## 11. Trực quan hóa t-SNE (Latent Space)

Plot phân bố các đội hình trong không gian ẩn $z_e$ sau khi được NCT encoder nhúng. Mỗi màu thể hiện một trường phái đội hình (Cluster) do VQ phân bổ.

In [ ]:
# Lấy subsample để t-SNE chạy nhanh và plot đẹp
SAMPLE_SIZE = min(5000, len(all_latent))
rng = np.random.RandomState(SEED)
sample_idx = rng.choice(len(all_latent), SAMPLE_SIZE, replace=False)

X_sample = all_latent[sample_idx]
y_sample = all_cluster_labels[sample_idx]

print(f"Đang chạy t-SNE trên {SAMPLE_SIZE} samples...")
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30)
X_tsne = tsne.fit_transform(X_sample)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], 
                      c=y_sample, cmap='tab10', s=12, alpha=0.6, edgecolors='none')
plt.colorbar(scatter, label='Cluster ID')
plt.title(f't-SNE: Không gian Ẩn Đội Hình LOL (NCT M={M_CLUSTERS}, Used M={used_M})', fontsize=13)
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.grid(True, alpha=0.15)
plt.tight_layout()
plt.show()

## 12. Counter Table Heatmap ($M \times M$)

Trực quan hóa bảng khắc chế $M \times M$ — giá trị trung bình của residual win value giữa các cặp cluster. Giá trị dương (đỏ) nghĩa là Cluster hàng "khắc chế" Cluster cột. Giá trị âm (xanh) nghĩa là bị khắc chế.

Theo paper Section 3.2.2, Eq. (8): $C_\theta(c_A, c_B) = C^d_\theta(C^e_\theta(c_A), C^e_\theta(c_B))$

In [ ]:
nct_model.eval()
with torch.no_grad():
    all_comp_A = all_features_tensor[:, 0, :].to(DEVICE)
    all_comp_B = all_features_tensor[:, 1, :].to(DEVICE)
    
    residual_all, _, _, _, idx_A_all, idx_B_all = nct_model(all_comp_A, all_comp_B)
    
    residual_np = residual_all.squeeze().cpu().numpy()
    idx_A_np = idx_A_all.squeeze().cpu().numpy()
    idx_B_np = idx_B_all.squeeze().cpu().numpy()

# Xây dựng M×M counter table
counter_table = np.zeros((M_CLUSTERS, M_CLUSTERS))
counter_count = np.zeros((M_CLUSTERS, M_CLUSTERS))

for i in range(len(idx_A_np)):
    a, b = int(idx_A_np[i]), int(idx_B_np[i])
    counter_table[a][b] += residual_np[i]
    counter_count[a][b] += 1

# Trung bình (tránh chia cho 0)
counter_table_avg = np.divide(counter_table, counter_count, 
                               out=np.zeros_like(counter_table), 
                               where=counter_count > 0)

plt.figure(figsize=(8, 7))
sns.heatmap(counter_table_avg, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            xticklabels=range(M_CLUSTERS), yticklabels=range(M_CLUSTERS),
            linewidths=0.5, square=True, cbar_kws={'label': 'Avg Residual Win Value'})
plt.title(f'Counter Table {M_CLUSTERS}×{M_CLUSTERS} — Mối quan hệ Khắc chế', fontsize=13)
plt.xlabel('Cluster Team B')
plt.ylabel('Cluster Team A')
plt.tight_layout()
plt.show()

# In bảng counter với số lượng matches
print(f"\nSố trận đấu trong từng ô của Counter Table:")
print(counter_count.astype(int))